# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')  # To clean output in notebook

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all `recordSet` objects, and for each:
- Show the `@id`, name, and description.
- List available fields (with their `@id` and names).

_Note: All Croissant entities (record sets, fields, columns, etc.) must be referenced by their `@id` fields._

In [ ]:
# Helper functions for traversing metadata
def find_record_sets(md):
    # RecordSets are under 'recordSet' or 'cr:recordSet'
    if 'recordSet' in md:
        recs = md['recordSet']
    elif 'cr:recordSet' in md:
        recs = md['cr:recordSet']
    else:
        recs = []
    # normalize to list
    if isinstance(recs, dict):
        recs = [recs]
    return recs

def find_field_obj(md, recset_obj):
    # returns a list of dicts for fields, will include `@id` and `name`
    flds = recset_obj.get('field') or recset_obj.get('cr:field')
    if flds is None:
        return []
    if isinstance(flds, dict):
        return [flds]
    return flds

recsets = find_record_sets(metadata)

if not recsets:
    print('No record sets listed in top-level metadata. Searching in schema...')
    # Try 'hasPart' for Croissant schemas that link datasets/files
    recsets = []
    if 'hasPart' in metadata:
        for part in metadata['hasPart']:
            if isinstance(part, dict) and (part.get('@type') == 'cr:RecordSet' or part.get('@type') == 'RecordSet'):
                recsets.append(part)
    if not recsets:
        # Try in distribution
        for file in metadata.get('distribution', []):
            if isinstance(file, dict) and (file.get('@type') == 'cr:RecordSet' or file.get('@type') == 'RecordSet'):
                recsets.append(file)

if not recsets:
    print('No record sets found in the metadata schema!')
else:
    for idx, recset in enumerate(recsets):
        print(f"RecordSet #{idx}")
        print('-'*40)
        print(f"RecordSet @id: {recset.get('@id', '<no_id>')}")
        print(f"Name: {recset.get('name', recset.get('schema:name', '<no_name>'))}")
        print(f"Description: {recset.get('description', recset.get('schema:description', '<no_description>'))}")

        fields = find_field_obj(metadata, recset)
        if not fields:
            print('  No fields listed.')
        else:
            print('  Fields:')
            for f in fields:
                print(f"    @id: {f.get('@id', '<no_id>')}, name: {f.get('name', f.get('schema:name', '<no_name>'))}")
        print()

# For this dataset, if no recordSets are printed above, the Croissant file is nonstandard, so we will use the default record set assumed by mlcroissant.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will list the record set `@id` and try to extract from it. If no explicit record sets are listed, mlcroissant will default to the main resource.

In [ ]:
# Try to extract all available record sets' IDs, use all if present, else fall back to None (=default)
recsets = find_record_sets(metadata)
if recsets:
    record_set_ids = [r['@id'] for r in recsets if '@id' in r]
else:
    # If not explicitly listed, dataset.records() with no argument defaults to only available one
    record_set_ids = [None]

dataframes = {}
for record_set_id in record_set_ids:
    # Use the @id if available, else use None
    key = record_set_id if record_set_id is not None else 'default'
    print(f"Loading records for record set @id: {key}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[key] = df
        print(f"  Loaded {len(df)} rows.")
    except Exception as e:
        print(f"  Failed to load records for @id {key}: {e}")

# List DataFrame columns for the first record set
first_key = record_set_ids[0] if record_set_ids[0] is not None else 'default'

if first_key in dataframes:
    print("Available columns (fields/columns, referenced by their @id when possible):")
    print(dataframes[first_key].columns.tolist())
    display(dataframes[first_key].head())
else:
    print("No dataframes loaded successfully.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

_We reference numeric and grouping fields using their `@id` where possible._

In [ ]:
# Choose the first loaded data frame for EDA.
import numpy as np

df_key = first_key
df = dataframes.get(df_key)

if df is not None:
    # Try to auto-identify a numeric field. Fallback to user input if unknown.
    numeric_candidates = [c for c in df.columns if df[c].dtype == float or df[c].dtype == int or np.issubdtype(df[c].dtype, np.number)]
    if not numeric_candidates:
        # Try to cast columns to float if possible (multi-source csvs may import as object)
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
            except Exception:
                continue
        numeric_candidates = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]

    if not numeric_candidates:
        print('No numeric field found; please examine the data and update `numeric_field_id` below.')
        numeric_field_id = df.columns[0]  # fallback
    else:
        numeric_field_id = numeric_candidates[0]  # first numeric column

    threshold = df[numeric_field_id].quantile(0.75) if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Add normalized column
    norm_col = f"{numeric_field_id}_normalized"
    if filtered_df[numeric_field_id].std() > 0:
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    else:
        filtered_df[norm_col] = 0
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to auto-identify a categorical/grouping field
    possible_groupby = [c for c in df.columns if df[c].nunique() < 20 and df[c].dtype == object]
    if possible_groupby:
        group_field = possible_groupby[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean values):")
        display(grouped_df.head())
    else:
        print("No grouping/categorical field found for groupby; skipping group analysis.")
else:
    print("No dataframe found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

_Below, we plot a histogram of the selected numeric field, and, if possible, a boxplot grouping by the selected group field._

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if df is not None:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by grouping field if found
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field_id, by=group_field, grid=False)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We have loaded the FAIR2 Croissant mass spectrometry dataset and viewed both its metadata and records.
- Fields/columns are referenced by their `@id` as per Croissant best practices.
- We identified numeric and grouping fields for filtering and EDA. Normalization and groupwise analysis were demonstrated.
- Distribution visualizations offer insight into the variance of protein abundance or other quantitative columns.

This notebook can now be adapted for further downstream analysis, such as protein annotation, significance testing, or data integration with other biomedical datasets.